In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Step 1: Install Dependencies**

In [ ]:
!pip install -q "transformers[torch]" datasets evaluate
!pip install -q git+https://github.com/csebuetnlp/normalizer
!pip install -q scikit-learn pandas matplotlib seaborn emoji

### **Step 2: Import Libraries and Mount Drive**

In [3]:
import os
import re
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
import matplotlib.pyplot as plt
import emoji

from google.colab import drive
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset, DatasetDict
from normalizer import normalize


### **Step 3: Define File Paths and Load Data**

In [ ]:
# --- Configuration ---
#MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
MODEL_NAME = "csebuetnlp/banglabert"
#MODEL_NAME = "FacebookAI/xlm-roberta-base"
# MODEL_NAME = "google/mt5-base"
# MODEL_NAME='sagorsarker/bangla-bert-base'
# MODEL_NAME = "intfloat/multilingual-e5-base"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- File Paths ---
TRAIN_FILE = "/content/drive/MyDrive/BLP Task/Task 1B/blp25_hatespeech_subtask_1B_train.tsv"
DEV_FILE = "/content/drive/MyDrive/BLP Task/Task 1B/blp25_hatespeech_subtask_1B_dev.tsv"
TEST_FILE = "/content/drive/MyDrive/BLP Task/Task 1B/blp25_hatespeech_subtask_1B_dev_test.tsv"
SUBMISSION_FILE = "subtask_1B.tsv"
ZIP_SUBMISSION_FILE = "submission.zip"

# --- Load Datasets ---
try:
    df_train = pd.read_csv(TRAIN_FILE, sep='\t', dtype=str, keep_default_na=False)
    df_dev = pd.read_csv(DEV_FILE, sep='\t', dtype=str, keep_default_na=False)
    df_test = pd.read_csv(TEST_FILE, sep='\t', dtype=str, keep_default_na=False)
    print("Files loaded successfully!")
    print(f"Train shape: {df_train.shape}")
    print(f"Dev shape: {df_dev.shape}")
    print(f"Test shape: {df_test.shape}")
except FileNotFoundError as e:
    print(f"Error: {e}. Please check your file paths in Google Drive.")

display(df_train.head())

### **Step 4: Text Preprocessing**
We use a robust cleaning function that normalizes Bengali unicode, demojizes emojis, and removes noise like URLs and HTML tags.

In [ ]:
def refined_preprocess_text(text):
    text = normalize(text)
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'([.?!,])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Preprocessing text data...")
df_train['cleaned_text'] = df_train['text'].apply(refined_preprocess_text)
df_dev['cleaned_text'] = df_dev['text'].apply(refined_preprocess_text)
df_test['cleaned_text'] = df_test['text'].apply(refined_preprocess_text)

print("\n--- Sample of Cleaned Text ---")
for i in range(3):
    print(f"Original: {df_train['text'][i]}")
    print(f"Cleaned:  {df_train['cleaned_text'][i]}\n")
print(df_train.columns)
print(df_dev.columns)
print(df_test.columns)

### **Step 5: Prepare Data for Hugging Face**
This involves label encoding, tokenizing the text, and creating `Dataset` objects.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Label Encoding
# Remove rows with NaN values in 'label' column
df_train_cleaned = df_train.dropna(subset=['label']).copy()
df_dev_cleaned = df_dev.dropna(subset=['label']).copy()

labels_list = sorted(df_train_cleaned['label'].unique())
# label2id = { "None": 0, "Religious Hate": 1, "Sexism": 2, "Political Hate": 3, "Profane": 4, "Abusive": 5, }

label2id = {
    'Society': 0,
    'Organization': 1,
    'None': 2,
    'Individual': 3,
    'Community': 4
}



id2label = {v: k for k, v in label2id.items()}
NUM_LABELS = len(labels_list)

print(f"Label to ID mapping: {label2id}")

df_train_cleaned['labels'] = df_train_cleaned['label'].map(label2id)
df_dev_cleaned['labels'] = df_dev_cleaned['label'].map(label2id)

# Create Hugging Face Datasets
train_dataset = Dataset.from_pandas(df_train_cleaned[['id','cleaned_text', 'labels']])
dev_dataset = Dataset.from_pandas(df_dev_cleaned[['id','cleaned_text', 'labels']])
test_dataset = Dataset.from_pandas(df_test[['id','cleaned_text']]) # Test set does not have labels

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': dev_dataset,
    'test': test_dataset
})

# Tokenization Function
def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], truncation=True, max_length=256)

print("\nTokenizing datasets...")
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Data Collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\n--- Final Prepared Datasets ---")
print(tokenized_datasets)

### **Step 6: TACT Implementation (FGM + Custom Trainer)**


In [ ]:
class FGM:
    """
    Fast Gradient Method (FGM) for adversarial training.
    Adds a perturbation to the model's embeddings.
    """
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=1.0, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                assert name in self.backup
                param.data = self.backup[name]
        self.backup = {}

class TACTTrainer(Trainer):
    """
    Custom Trainer that incorporates FGM for TACT.
    The total loss is the sum of the original loss and the adversarial loss.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm = FGM(self.model)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # --- Standard Forward Pass ---
        outputs = model(**inputs)
        loss = outputs.loss

        # --- Adversarial Training Step ---
        # Only perform adversarial training if the model is in training mode
        if model.training:
            loss.backward(retain_graph=True)

            # 1. FGM Attack
            # NOTE: name might change for other models, inspect model.electra.embeddings.word_embeddings.named_parameters()
            self.fgm.attack(emb_name='electra.embeddings.word_embeddings')

            # 2. Compute loss on adversarial example
            adv_outputs = model(**inputs)
            adv_loss = adv_outputs.loss

            # 3. Restore original embeddings
            self.fgm.restore(emb_name='electra.embeddings.word_embeddings')

            # 4. Combine losses
            loss = loss + adv_loss

        return (loss, outputs) if return_outputs else loss

print("TACT components (FGM, TACTTrainer) are defined.")

### **Step 7: LLRD Optimizer Setup**
We keep the Layer-wise Learning Rate Decay optimizer, as it is a powerful technique for fine-tuning Transformer models and complements TACT well.

In [ ]:
def create_llrd_optimizer(model, learning_rate=2e-5, layer_decay=0.95):
    optimizer_grouped_parameters = []
    no_decay = ["bias", "LayerNorm.weight"]

    # Detect backbone automatically (bert, roberta, electra, etc.)
    if hasattr(model, "roberta"):
        base_model = model.roberta
    elif hasattr(model, "bert"):
        base_model = model.bert
    elif hasattr(model, "electra"):
        base_model = model.electra
    elif hasattr(model, "xlm_roberta"):
        base_model = model.xlm_roberta
    else:
        base_model = model.base_model  

    num_layers = base_model.config.num_hidden_layers

    # ----- Classifier head -----
    optimizer_grouped_parameters.extend([
        {
            "params": [p for n, p in model.classifier.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            "weight_decay": 0.01,
            "lr": learning_rate,
        },
        {
            "params": [p for n, p in model.classifier.named_parameters()
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": learning_rate,
        },
    ])

    # ----- Transformer encoder layers -----
    for i in range(num_layers - 1, -1, -1):
        layer_lr = learning_rate * (layer_decay ** (num_layers - 1 - i))
        optimizer_grouped_parameters.extend([
            {
                "params": [p for n, p in base_model.encoder.layer[i].named_parameters()
                           if not any(nd in n for nd in no_decay)],
                "weight_decay": 0.01,
                "lr": layer_lr,
            },
            {
                "params": [p for n, p in base_model.encoder.layer[i].named_parameters()
                           if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
                "lr": layer_lr,
            },
        ])

    # ----- Embedding layer -----
    embedding_lr = learning_rate * (layer_decay ** num_layers)
    optimizer_grouped_parameters.extend([
        {
            "params": [p for n, p in base_model.embeddings.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            "weight_decay": 0.01,
            "lr": embedding_lr,
        },
        {
            "params": [p for n, p in base_model.embeddings.named_parameters()
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": embedding_lr,
        },
    ])

    return torch.optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)

print(" LLRD optimizer function defined (works with XLM-R, BERT, RoBERTa, ELECTRA, etc.).")


### **Step 8: Model Training**
We configure the training arguments and instantiate our new `TACTTrainer` to begin training.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import torch

# Load model with correct head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,       # your 6-class task
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  
).to(DEVICE)

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",   # <-- correct param name
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=1,
)

# Optimizer (LLRD/AdamW)
optimizer = create_llrd_optimizer(model, learning_rate=training_args.learning_rate)

# Trainer
trainer = TACTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)

print("Starting model training with TACT and LLRD...")
trainer.train()
print("\nTraining finished.")

### **Step 9: Final Evaluation**
We evaluate the best model checkpoint on the development set.

In [ ]:
print("Evaluating the best model on the validation set...")
eval_results = trainer.evaluate()

print("\n--- Final Evaluation Results ---")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
print(tokenized_datasets["test"])

### **Step 10: Prediction and Submission**
Generate predictions on the test set and create the `submission.zip` file.

In [14]:
print("Making predictions on the test set...")
test_dataset_formatted = tokenized_datasets["test"].remove_columns([col for col in ['labels', 'label'] if col in tokenized_datasets["test"].column_names])
ids_2 = tokenized_datasets["test"]['id']
test_predictions = trainer.predict(test_dataset_formatted)
preds = test_predictions.predictions
#print(ids_2)

Making predictions on the test set...


In [ ]:
print(test_predictions)

In [17]:
LABEL2ID = {
    'Society': 0,
    'Organization': 1,
    'None': 2,
    'Individual': 3,
    'Community': 4
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
logits_list_2 = [list(row) for row in preds]
#model_name = 'Custom'

In [18]:
output_predict_file = os.path.join(training_args.output_dir, f"subtask_1B_pred.tsv")
if trainer.is_world_process_zero():
    with open(output_predict_file, "w") as writer:
        #logger.info(f"***** Predict results *****")
        writer.write("id\tlogits\tmodel\n")
        for index, item in enumerate(logits_list_2):
            # item = id2l[item]
            #writer.write(f"{ids[index]}\t{item}\t{model_name}\n")
            writer.write(f"{ids_2[index]}\t{' '.join(map(str, item))}\t{MODEL_NAME}\n")